<a href="https://colab.research.google.com/github/Hadi6618/Fusion/blob/main/ShanghaiTech_Ensemble_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STG-NF + MULDE Frame-Level Ensemble Fusion

This notebook fuses frame-level anomaly scores from **STG-NF** (object/pose level)
and **MULDE** (frame level) for either the **ShanghaiTech Campus** or **Avenue**
dataset.

**Pipeline:**
1. Load the two score pickles produced by the upstream training/export notebooks.
2. Align the two streams per `(video_id, frame_index)`. Video-ID conventions are
   auto-detected and remapped (Avenue: STG-NF `01_0001` <-> MULDE `01`).
3. Apply **global rank** normalization so both streams live on `[0, 1]`.
4. Grid-search the best Gaussian smoothing `sigma` (maximises average standalone AUC).
5. Grid-search the fusion weight `beta_1` over `[0, 1]` in 0.01 steps.
6. Save the grid table (`fusion_grid_search.csv`) and report (`fusion_report.json`)
   to Google Drive.

**Set `DATASET` in the config cell to choose the dataset.** All results are saved
under `/content/drive/MyDrive/Fusion/runs/<DATASET>/ensemble/`.

In [ ]:
import subprocess
import importlib.util
import sys
import warnings
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/Hadi6618/Fusion.git'
REPO_DIR = Path('/content/Fusion')


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR} already exists, skipping clone.')


In [ ]:
_FUSION_PATH = REPO_DIR / 'fusion.py'
if not _FUSION_PATH.exists():
    raise FileNotFoundError(f'fusion.py not found at {_FUSION_PATH}')
spec = importlib.util.spec_from_file_location('fusion', _FUSION_PATH)
fusion = importlib.util.module_from_spec(spec)
sys.modules['fusion'] = fusion
spec.loader.exec_module(fusion)
print(f'Loaded fusion module from {_FUSION_PATH}')


## Dataset selection

Set `DATASET` to `'ShanghaiTech'` or `'Avenue'`. The paths for each dataset are
pre-configured below — edit them only if your Drive layout differs.

In [ ]:
# ======================================================================
# DATASET SELECTOR -- change this one line to switch datasets
# ======================================================================
DATASET = 'ShanghaiTech'   # 'ShanghaiTech' or 'Avenue'

# Per-dataset paths on Google Drive. Edit if your layout differs.
DATASET_PATHS = {
    'ShanghaiTech': {
        'stgnf_pkl': Path('/content/drive/MyDrive/STG-NF/original_shanghaitech/logs/shanghaitech_stgnf_scores_84.pkl'),
        'mulde_pkl':  Path('/content/drive/MyDrive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/shanghaitech_mulde_scores_79_7.pkl'),
        'output_dir': Path('/content/drive/MyDrive/Fusion/runs/ShanghaiTech/ensemble'),
    },
    'Avenue': {
        'stgnf_pkl': Path('/content/drive/MyDrive/STG-NF/Avenue_dataset/logs/avenue_stgnf_scores_57.pkl'),
        'mulde_pkl':  Path('/content/drive/MyDrive/MULDE/runs/avenue_hiera_l_mulde/Final_avenue_scores/artifacts/avenue_mulde_scores_81_4.pkl'),
        'output_dir': Path('/content/drive/MyDrive/Fusion/runs/Avenue/ensemble'),
    },
}

assert DATASET in DATASET_PATHS, f'DATASET must be one of {list(DATASET_PATHS.keys())}'
cfg = DATASET_PATHS[DATASET]
STGNF_PKL = cfg['stgnf_pkl']
MULDE_PKL = cfg['mulde_pkl']
OUTPUT_DIR = cfg['output_dir']

for p in [STGNF_PKL, MULDE_PKL]:
    if not p.exists():
        raise FileNotFoundError(f'Missing score file: {p}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset:      {DATASET}')
print(f'STG-NF scores: {STGNF_PKL}')
print(f'MULDE scores:  {MULDE_PKL}')
print(f'Output dir:    {OUTPUT_DIR}')


In [ ]:
stgnf_scores, stgnf_meta = fusion.load_score_pickle(STGNF_PKL)
mulde_scores, mulde_meta = fusion.load_score_pickle(MULDE_PKL)
print(f'STG-NF videos: {len(stgnf_scores)}  (sample IDs: {list(stgnf_scores.keys())[:3]})')
print(f'MULDE  videos: {len(mulde_scores)}  (sample IDs: {list(mulde_scores.keys())[:3]})')


In [ ]:
# Align per (video_id, frame_index). auto_detect_offset sweeps offset in {-2..+2}
# and tries both anomaly/normality polarity for STG-NF, picking the combination
# that maximises STG-NF's standalone Micro AUC. Video-ID aliases are applied
# automatically (Avenue: 01_0001 <-> 01).
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    aligned, align_stats = fusion.align_per_video(
        stgnf_scores,
        mulde_scores,
        auto_detect_offset=True,
        stgnf_score_mode='auto',
    )

n_remap = align_stats.get('video_id_mapping_applied', 0)
print(
    f"Aligned videos: {align_stats['videos_aligned']} "
    f"(STG-NF={align_stats['videos_in_stgnf']}, MULDE={align_stats['videos_in_mulde']})"
)
print(f"  stgnf_frame_offset = {align_stats.get('stgnf_frame_offset', 0)}")
print(f"  stgnf_score_mode   = {align_stats.get('stgnf_score_mode', 'auto')}")
print(f"  video_ids_remapped = {n_remap}")
print(f"  label_inversion    = {align_stats.get('label_inversion_detected', False)}")
if 'auto_detect' in align_stats:
    print('  Offset/polarity candidates:')
    for key, payload in align_stats['auto_detect']['stgnf_micro_auc_per_candidate'].items():
        off, mode = key.split('|', 1)
        auc = payload.get('micro_auc_stgnf')
        auc_s = f'{auc * 100:.4f}%' if auc is not None else 'n/a'
        marker = ' <-- chosen' if (
            int(off) == align_stats.get('stgnf_frame_offset', 0)
            and mode == align_stats.get('stgnf_score_mode', 'auto')
        ) else ''
        print(f'    offset={int(off):+d}  mode={mode:9s}  AUC={auc_s}{marker}')


In [ ]:
# Global rank normalization: converts each model's raw scores to [0,1] percentiles.
# Must happen BEFORE Gaussian smoothing so outlier spikes are capped at 1.0.
NORMALIZATION = 'global_rank'
aligned = fusion.apply_normalization(aligned, strategy=NORMALIZATION)
print(f'Normalization strategy: {NORMALIZATION}')

_y = np.concatenate([v.labels for v in aligned])
_s = np.concatenate([v.stgnf_scores for v in aligned])
_m = np.concatenate([v.mulde_scores for v in aligned])
if len(np.unique(_y)) >= 2:
    print(f'STG-NF alone (after norm) Micro AUC: {roc_auc_score(_y, _s) * 100:.4f}%')
    print(f'MULDE  alone (after norm) Micro AUC: {roc_auc_score(_y, _m) * 100:.4f}%')


In [ ]:
print('Searching for best Gaussian smoothing sigma ...')
best_sigma, sigma_results = fusion.search_best_sigma(
    aligned,
    sigma_candidates=(0, 1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15),
    normalization=None,  # already normalized above
)
print(f'Best sigma = {best_sigma}')
aligned = fusion.smooth_scores(aligned, sigma=best_sigma)


In [ ]:
beta_1_values = list(np.round(np.arange(0.0, 1.0 + 1e-9, 0.01), 4))
results, best, summary = fusion.grid_search_fusion(aligned, beta_1_values=beta_1_values)
summary['dataset'] = DATASET
summary['normalization'] = NORMALIZATION
summary['smooth_sigma'] = best_sigma
fusion.write_outputs(
    results=results,
    best=best,
    summary=summary,
    alignment_stats=align_stats,
    stgnf_meta=stgnf_meta,
    mulde_meta=mulde_meta,
    output_dir=OUTPUT_DIR,
)


In [ ]:
import pandas as pd
table = fusion.results_to_table(results)
table_sorted = table.dropna(subset=['micro_auc']).sort_values('micro_auc', ascending=False)
display(table_sorted.head(30))


In [ ]:
y = np.concatenate([v.labels for v in aligned])
s = np.concatenate([v.stgnf_scores for v in aligned])
m = np.concatenate([v.mulde_scores for v in aligned])
print(f'Dataset:    {DATASET}')
print(f'Frames:     {y.shape[0]}')
print(f'Videos:     {len(aligned)}')
print(f'STG-NF AUC: {roc_auc_score(y, s) * 100:.4f}%')
print(f'MULDE  AUC: {roc_auc_score(y, m) * 100:.4f}%')
print(f'Sigma:      {best_sigma}')
if best is not None and best.micro_auc is not None:
    print(f'Ensemble:   {best.micro_auc * 100:.4f}% (beta_1={best.beta_1:.2f}, beta_2={best.beta_2:.2f})')
print(f'Correlation (STG-NF vs MULDE): {np.corrcoef(s, m)[0, 1]:.4f}')
